CELL 1 — PROJECT TITLE

###  AI Drowsiness Detection System
#### Task 2.3 — Adaptive Threshold & Alert System

This notebook implements a real-time drowsiness detection system using:

- MediaPipe Face Mesh
- Eye Aspect Ratio (EAR)
- Adaptive Threshold Calibration
- Timer-Based Eye Closure Detection
- Audio Alarm System

The system dynamically adjusts the EAR threshold based on the user's eye characteristics for improved accuracy.

CELL 2 — OBJECTIVE

#### Objective

The objective of this task is to:

- Detect eye closure in real time
- Calculate Eye Aspect Ratio (EAR)
- Adapt the threshold to different users
- Differentiate blinking from drowsiness
- Trigger an alarm when prolonged eye closure is detected

This system improves traditional fixed-threshold detection by introducing adaptive threshold calibration.

CELL 3 — IMPORT LIBRARIES

In [208]:
import cv2
import mediapipe as mp
import numpy as np
import os
import threading
import time

from scipy.spatial import distance
from playsound import playsound

CELL — ADAPTIVE THRESHOLD VARIABLES

In [209]:
# -----------------------------------
# ADAPTIVE THRESHOLD VARIABLES
# -----------------------------------

calibration_mode = True

calibration_frames = 100
calibration_count = 0

ear_samples = []

average_ear = 0
adaptive_threshold = 0

calibration_display_start = None
show_calibration_result = False

CELL 4 — INITIALIZE MEDIAPIPE

In [210]:
# Initialize MediaPipe Face Mesh

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    refine_landmarks=True,
    max_num_faces=1
)

mp_draw = mp.solutions.drawing_utils

CELL 5 — EYE LANDMARK INDICES

In [211]:
# Left Eye Landmark Indices
LEFT_EYE = [33, 160, 158, 133, 153, 144]

# Right Eye Landmark Indices
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

CELL 6 — EAR FUNCTION

In [212]:
# Eye Aspect Ratio (EAR) Function

def calculate_EAR(eye_points):

    # Vertical distances
    vertical_1 = distance.euclidean(eye_points[1], eye_points[5])
    vertical_2 = distance.euclidean(eye_points[2], eye_points[4])

    # Horizontal distance
    horizontal = distance.euclidean(eye_points[0], eye_points[3])

    # EAR formula
    ear = (vertical_1 + vertical_2) / (2.0 * horizontal)

    return ear

CELL 7 — ALARM FUNCTION

In [213]:
# Alarm Function

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

ALARM_PATH = os.path.join(BASE_DIR, "assets", "audio", "alarm.wav")

alarm_playing = False


def play_alarm():

    global alarm_playing

    if not alarm_playing:

        alarm_playing = True

        try:
            playsound(ALARM_PATH)

        except Exception as e:
            print("Alarm Error:", e)

        alarm_playing = False

CELL 8 — ADAPTIVE THRESHOLD VARIABLES

In [214]:
# Adaptive Threshold Variables

EAR_THRESHOLD = 0

CLOSED_SECONDS = 4

calibration_start = time.time()

CALIBRATION_TIME = 5

ear_values = []

calibrated = False

eye_closed_start = None

CELL 9 — MAIN DETECTION LOOP

In [215]:
# =========================================
# MAIN DETECTION LOOP
# =========================================

cap = cv2.VideoCapture(0)

print("System Started...")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    # Flip frame
    frame = cv2.flip(frame, 1)

    # Convert to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process Face Mesh
    results = face_mesh.process(rgb_frame)

    frame_height, frame_width, _ = frame.shape

    current_time = time.time()

    if results.multi_face_landmarks:

        for face_landmarks in results.multi_face_landmarks:

            left_eye_points = []
            right_eye_points = []

            # =========================================
            # LEFT EYE LANDMARKS
            # =========================================

            for index in LEFT_EYE:

                landmark = face_landmarks.landmark[index]

                x = int(landmark.x * frame_width)
                y = int(landmark.y * frame_height)

                left_eye_points.append((x, y))

                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            # =========================================
            # RIGHT EYE LANDMARKS
            # =========================================

            for index in RIGHT_EYE:

                landmark = face_landmarks.landmark[index]

                x = int(landmark.x * frame_width)
                y = int(landmark.y * frame_height)

                right_eye_points.append((x, y))

                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            # =========================================
            # CALCULATE EAR
            # =========================================

            left_ear = calculate_EAR(left_eye_points)

            right_ear = calculate_EAR(right_eye_points)

            ear = (left_ear + right_ear) / 2

            # =========================================
            # CALIBRATION PHASE
            # =========================================

            if calibration_mode:

                ear_samples.append(ear)

                calibration_count += 1

                cv2.putText(
                    frame,
                    "CALIBRATING... KEEP EYES OPEN",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 255),
                    2
                )

                cv2.putText(
                    frame,
                    f"Collecting Samples: {calibration_count}/{calibration_frames}",
                    (30, 90),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Current EAR: {ear:.3f}",
                    (30, 130),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2
                )

                # Finish calibration
                if calibration_count >= calibration_frames:

                    average_ear = np.mean(ear_samples)

                    adaptive_threshold = average_ear * 0.75

                    calibration_mode = False

                    show_calibration_result = True

                    calibration_display_start = time.time()

            # =========================================
            # SHOW CALIBRATION RESULT
            # =========================================

            elif show_calibration_result:

                elapsed_time = time.time() - calibration_display_start

                cv2.putText(
                    frame,
                    "CALIBRATION COMPLETE",
                    (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9,
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Average EAR: {average_ear:.3f}",
                    (30, 120),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Adaptive Threshold: {adaptive_threshold:.3f}",
                    (30, 170),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 255),
                    2
                )

                cv2.putText(
                    frame,
                    "Starting Monitoring...",
                    (30, 230),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

                # Keep on screen for 7 seconds
                if elapsed_time >= 7:

                    show_calibration_result = False

            # =========================================
            # MAIN DETECTION STAGE
            # =========================================

            else:

                cv2.putText(
                    frame,
                    f"EAR: {ear:.3f}",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Threshold: {adaptive_threshold:.3f}",
                    (30, 90),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 0),
                    2
                )

                # =========================================
                # EYES CLOSED DETECTION
                # =========================================

                if ear < adaptive_threshold:

                    if eye_closed_start is None:

                        eye_closed_start = time.time()

                    closed_duration = time.time() - eye_closed_start

                    cv2.putText(
                        frame,
                        f"Eyes Closed: {closed_duration:.1f}s",
                        (30, 130),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (0, 0, 255),
                        2
                    )

                    # =========================================
                    # DROWSINESS ALERT
                    # =========================================

                    if closed_duration >= CLOSED_SECONDS:

                        cv2.putText(
                            frame,
                            "DROWSINESS ALERT!",
                            (150, 220),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0, 0, 255),
                            3
                        )

                        threading.Thread(
                            target=play_alarm,
                            daemon=True
                        ).start()

                else:

                    eye_closed_start = None

    # =========================================
    # SHOW WINDOW
    # =========================================

    cv2.imshow("Adaptive Drowsiness Detection", frame)

    # Press Q to Quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# =========================================
# CLEANUP
# =========================================

cap.release()

cv2.destroyAllWindows()

System Started...


In [216]:
# Cleanup

cap.release()

cv2.destroyAllWindows()